In [1]:
# conda activate genomic_tools

import os
import glob
import json
import pickle
import pandas as pd
from collections import defaultdict

pd.set_option('display.max_columns', None)

## Load UCSC genome browser tracks

In [103]:
track_df_list = []
keep = ["unipLocTransMemb", "unipOther", "unipLocSignal", "unipModif", "unipRepeat", "unipLocCytopl", "unipChain", "unipLocExtra", "unipDomain", "unipStruct", "unipDisulfBond", "unipInterest"]
track_dir = "/mnt/lareaulab/reliscu/data/UCSC/hg38/genome_tracks/uniprot"

for file in glob.glob(f"{track_dir}/*.bed"):
    file_name = file.split("/")[-1].split(".bb.bed")[0]
    if file_name in keep:
        track_df = pd.read_csv(file, sep="\t")
        track_df.insert(0, "file_name", file_name)
        track_df_list.append(track_df)

/tmp/ipykernel_2168317/1040619109.py:8: DtypeWarning: Columns (0: pmids) have mixed types. Specify dtype option on import or set low_memory=False.
  track_df = pd.read_csv(file, sep="\t")
/tmp/ipykernel_2168317/1040619109.py:8: DtypeWarning: Columns (0: longName, 1: syns) have mixed types. Specify dtype option on import or set low_memory=False.
  track_df = pd.read_csv(file, sep="\t")


## Load interproscan results

In [3]:
cols = [
    'protein_accession',
    'sequence_md5',
    'sequence_length',
    'analysis',
    'signature_accession',
    'signature_description',
    'start',
    'stop',
    'score',
    'status',
    'date',
    'interpro_accession',
    'interpro_description',
    'go_annotations',
    'pathways'
]

interproscan_results = pd.read_csv(
    "data/interproscan/results/proteins.fa.tsv",
    sep='\t',
    header=None,
    names=cols,
    index_col=False
)

In [4]:
# Parse the PIRSR data

with open("data/interproscan/interpro/data/pirsr/sr_uru.json") as f:
    pirsr_data = json.load(f)

# subset to proteins patterns applicable to humans
human_relevant = ['Eukaryota', 'Eukaryota; Metazoa', 'Eukaryota; Vertebrata', 'Eukaryota; Chordata', 'Eukaryota; Mammalia', 'Eukaryota; Eutheria']

records = []
for ac, entry in pirsr_data.items():
    for group_id, sites in entry['Groups'].items():
        for site in sites:
            scope = entry.get('Scope', [])
            tr = entry.get('TR', '')
            if any(s in human_relevant for s in scope):
                records.append({
                    'accession': ac,
                    'scope':  ', '.join(scope),
                    'TR': tr.split("; ")[1],
                    'label': site['label'],
                    'condition': site['condition'],
                    'desc': site['desc'],
                    'group': group_id
                })

pirsr_df = pd.DataFrame(records)

pirsr_df = pirsr_df.groupby("accession").agg(
    scope=('scope', lambda x: ' | '.join(x.unique())),
    TR=('TR', lambda x: ' | '.join(x.unique())),
    label=('label', lambda x: ' | '.join(x.unique())),
    condition=('condition', lambda x: ' | '.join(x.unique())),
    desc=('desc', lambda x: ' | '.join(x.unique())),
    group=('group', lambda x: ' | '.join(x.unique())),
).reset_index()

In [5]:
interproscan_results = interproscan_results.merge(pirsr_df, left_on="signature_accession", right_on="accession", how="left")

interproscan_results.loc[interproscan_results['analysis'] == "PIRSR", 'signature_description'] = interproscan_results.loc[interproscan_results['analysis'] == "PIRSR", 'label']
interproscan_results.loc[interproscan_results['analysis'] == "DeepTMHMM", 'signature_description'] = interproscan_results.loc[interproscan_results['analysis'] == "DeepTMHMM", 'signature_accession']
interproscan_results.loc[interproscan_results['analysis'] == "TMbed", 'signature_description'] = interproscan_results.loc[interproscan_results['analysis'] == "TMbed", 'signature_accession']  

In [6]:
interproscan_results.shape

(628144, 22)

## Map events to domains

In [8]:
def near_exon(exon_start, exon_end, scan_start, scan_end, window=600):
    """Returns boolean mask for features within window of the exon."""
    return (
        (exon_end  >= scan_start - window) &
        (exon_start <= scan_end   + window)
    )
    
def _cds_rows(obj):
    if obj is None:
        return None
    if hasattr(obj, "iterrows"):
        return [{'start': int(r['start']), 'end': int(r['end']), 'frame': int(r['frame'])}
                for _, r in obj.iterrows()]
    return [{'start': int(c['start']), 'end': int(c['end']), 'frame': int(c['frame'])} for c in obj]

def rel_cds_to_genome(cds_obj, strand):
    """
    Build a list of exon segments mapping between genomic coordinates
    and CDS-relative coordinates, in transcript (5'->3') order.

    Each segment is a dict:
        genome_start, genome_end : genomic coordinates (always genome_start <= genome_end)
        rel_start, rel_end       : CDS-relative coordinates (always rel_start <= rel_end)
    """
    cds = _cds_rows(cds_obj)

    # order exons in transcript (5'->3') order
    if strand == '-':
        cds = sorted(cds, key=lambda c: c['start'], reverse=True)
    else:
        cds = sorted(cds, key=lambda c: c['start'])

    segments = []
    cds_start = 0
    for c in cds:
        length = c['end'] - c['start'] + 1
        segments.append({
            'genome_start': c['start'],
            'genome_end': c['end'],
            'rel_start': cds_start,
            'rel_end': cds_start + length - 1,
        })
        cds_start += length

    return segments

def map_aa_to_genome(rel_to_genome, aa_start, aa_end, strand):
    result = []
    # convert AA position to relative CDS position
    # note: subtract 1 to convert from 1-indexed AA to 0-indexed
    cds_start = (aa_start - 1) * 3
    cds_end = (aa_end - 1) * 3 + 2
    for seg in rel_to_genome:
        # seg contains mapping from relative CDS position to genome position
        overlap_start = max(cds_start, seg['rel_start'])
        overlap_end = min(cds_end, seg['rel_end'])
        if overlap_start <= overlap_end:
            if strand == '-':
                # rel increases as genome decreases
                g_start = seg['genome_end'] - (overlap_end - seg['rel_start'])
                g_end = seg['genome_end'] - (overlap_start - seg['rel_start'])
            else:
                # rel increases as genome increases
                g_start = seg['genome_start'] + (overlap_start - seg['rel_start'])
                g_end = seg['genome_start'] + (overlap_end - seg['rel_start'])
            result.append((g_start, g_end))
    return result

In [ ]:
with open('data/event_protein_map.pkl', 'rb') as f:
    event_protein_map = pickle.load(f)

In [ ]:

import pandas as pd

FLANK = 600

def flatten_exon_data(data: dict, flank=600) -> pd.DataFrame:
    rows = []
    for ev, rec in data.items():
        meta = rec["meta"]

        def emit(type_name, d, start=None, end=None):
            row = {
                "event_id": ev,
                "chrom": meta["chrom"],
                "strand": meta["strand"],
                "gene": meta["gene"],
                "meta_es": meta["es"],
                "meta_ee": meta["ee"],
                "type": type_name,
            }
            row.update(d)  # transcript_id, exon_cds_start/end, frame_preserving, etc.
            row["start"] = start
            row["end"] = end
            rows.append(row)
        
        # inclusion: use its own exon_cds_start/end
        for type_name in ("inclusion", "exon_diff_boundary_siblings", "exon_diff_junction_siblings"):
            d = rec.get(type_name)
            if d and d.get("transcript_id") is not None:
                emit(type_name, d, 
                     start=d["exon_cds_start"] - flank, 
                     end=d["exon_cds_end"] + flank)

        # real_skip: no exon coords (it's skipped) — use flanked window around the parent exon
        d = rec.get("real_skip")
        if d and d.get("transcript_id") is not None:
            emit("real_skip", d,
                 start=meta["es"] - flank,
                 end=meta["ee"] + flank)
        
    df = pd.DataFrame(rows)
    df["start"] = df["start"].astype(int)
    df["end"] = df["end"].astype(int)

    return df

long_df = flatten_exon_data(event_protein_map)

#### Merge Uniprot tracks with splicing events

In [ ]:
import bioframe as bf

cols = [
    "file_name", "chrom", "chromStart", "chromEnd", "name", "status", "annotationType", "position"
]

results = []
for track_df in track_df_list:
    # t = track_df.rename(columns={"chromStart": "start", "chromEnd": "end"})
    t = track_df.loc[:, cols]
    overlapped = bf.overlap(
        long_df, t,
        cols1=("chrom", "start", "end"),
        cols2=("chrom", "chromStart", "chromEnd"),
        suffixes=("", "_") 
    )
    results.append(overlapped)

uniprot_merged = pd.concat(results, ignore_index=True)

In [ ]:
uniprot_by_event = {
    event_id: grp
    for event_id, grp in uniprot_merged.groupby("event_id")  # adjust column name — see note below
}

In [ ]:
pickle.dump(uniprot_by_event, open("data/uniprot_by_event.pkl", "wb"))

#### Merge IntroProScan results

In [ ]:
with open("data/gencode.v46.annotation_cds_by_transcript.pkl", "rb") as f:
    cds_by_transcript = pickle.load(f)

In [ ]:
# map interproscan results to their corresponding transcript (representing the splicing event of interest)
cols = ['protein_accession', 'sequence_length', 'analysis', 
        'signature_description', 'start', 'stop', 'interpro_description']
ipr = interproscan_results.loc[:, columns]

ipr_grouped = {acc: grp for acc, grp in ipr.groupby('protein_accession')}

interproscan_by_event = defaultdict(dict)

for ev, rec in event_protein_map.items():
    
    rec_incl = rec['inclusion']
    incl_df  = ipr_grouped.get(rec_incl['transcript_id'])
    if incl_df is None:
        continue
    
    overlap_mask = [
        near_exon(
            rec_incl['aa_start'], 
            rec_incl['aa_end'], 
            row['start'], 
            row['stop'],
            window=200
        ) 
        for _, row in incl_df.iterrows()
    ]
    overlap_df = incl_df[overlap_mask]
    
    if not overlap_df.empty:
        strand = rec['meta']['strand']

        # save AA position of protein domains in terms of genomic coordinates
        cds_obj = cds_by_transcript[rec_incl['transcript_id']]
        cds_to_genome = rel_cds_to_genome(cds_obj, strand)
        
        genome_coords = []
        for _, row in overlap_df.iterrows():
            aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
            genome_coords.append(aa_to_genome)
    
        incl_cols = ['aa_start', 'aa_end', 
                     'exon_cds_start', 'exon_cds_end',
                     'frame_preserving', 'clean_start', 'clean_end']
        
        event_interproscan_map[ev] = {
            'meta': rec['meta'],
            'inclusion': overlap_df.assign(
                **{col: rec_incl[col] for col in incl_cols},
                genome_coords=genome_coords
            ).reset_index(drop=True),
            'real_skip': None,
            'exon_diff_junction_siblings': None,
            'exon_diff_boundary_siblings': None
        }
    
        # real skip
        if rec.get('real_skip'):
            skip_df = ipr_grouped.get(rec['real_skip'])
            if skip_df is not None: 
                cds_obj = cds_by_transcript[rec['real_skip']]
                cds_to_genome = rel_cds_to_genome(cds_obj, strand)
                genome_coords = []
                for _, row in skip_df.iterrows():
                    # real_skip: no exon coords (it's skipped) — use flanked window around the parent exon
                    # note: if real skip transcript is very different 5' structure, the original exon neighborhood may be out of scope
                    aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
                    scan_start = min(min(t) for t in aa_to_genome)
                    scan_end = max(max(t) for t in aa_to_genome)
                    if near_exon(rec_incl['exon_cds_start'], rec_incl['exon_cds_end'], scan_start, scan_end): 
                        genome_coords.append(aa_to_genome)
                    else:
                        genome_coords.append(None)
                        
                skip_df = skip_df.assign(genome_coords=genome_coords)
                skip_df = skip_df[skip_df['genome_coords'].notna()]
                    
                if not skip_df.empty:
                    interproscan_by_event[ev]['real_skip'] = skip_df.reset_index(drop=True)
                    
        # junction siblings
        if rec.get('exon_diff_junction_siblings'):
            sib_frames = []
            for sib in rec['exon_diff_junction_siblings']:
                t = sib['transcript_id']
                sib_df = ipr_grouped.get(t) 
                if sib_df is None:
                    continue
                
                overlap_mask = [
                    near_exon(
                        sib['aa_start'], 
                        sib['aa_end'], 
                        row['start'], 
                        row['stop'],
                        window=200
                    ) 
                    for _, row in sib_df.iterrows()
                ]
                
                sib_overlap = sib_df[overlap_mask]
                
                if not sib_overlap.empty:
                    # save AA position of protein domains in terms of genomic coordinates
                    cds_obj = cds_by_transcript[t]
                    cds_to_genome = rel_cds_to_genome(cds_obj, strand)
                    genome_coords = []
                    for _, row in sib_overlap.iterrows():
                        aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
                        genome_coords.append(aa_to_genome)

                    sib_frames.append(sib_overlap.assign(
                        aa_start=sib['aa_start'],
                        aa_end=sib['aa_end'],
                        exon_cds_start=sib['exon_cds_start'],
                        exon_cds_end=sib['exon_cds_end'],
                        genome_coords=genome_coords
                    ).reset_index(drop=True))

            if sib_frames:
                interproscan_by_event[ev]['exon_diff_junction_siblings'] = pd.concat(sib_frames, ignore_index=True)

        # boundary siblings
        if rec.get('exon_diff_boundary_siblings'):
            sib_frames = []
            for sib in rec['exon_diff_boundary_siblings']:
                t = sib['transcript_id']
                sib_df = ipr_grouped.get(t) 
                if sib_df is None:
                    continue
                
                overlap_mask = [
                    near_exon(
                        sib['aa_start'], 
                        sib['aa_end'], 
                        row['start'], 
                        row['stop'],
                        window=200
                    ) 
                    for _, row in sib_df.iterrows()
                ]
                
                sib_overlap = sib_df[overlap_mask]
                
                if not sib_overlap.empty:
                    # save AA position of protein domains in terms of genomic coordinates
                    cds_obj = cds_by_transcript[t]
                    cds_to_genome = rel_cds_to_genome(cds_obj, strand)
                    genome_coords = []
                    for _, row in sib_overlap.iterrows():
                        aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
                        genome_coords.append(aa_to_genome)

                    sib_frames.append(sib_overlap.assign(
                        aa_start=sib['aa_start'],
                        aa_end=sib['aa_end'],
                        exon_cds_start=sib['exon_cds_start'],
                        exon_cds_end=sib['exon_cds_end'],
                        genome_coords=genome_coords
                    ).reset_index(drop=True))

            if sib_frames:
                interproscan_by_event[ev]['exon_diff_boundary_siblings'] = pd.concat(sib_frames, ignore_index=True)

In [ ]:
pickle.dump(interproscan_by_event, open("data/interproscan_by_event.pkl", "wb"))

### Debug

In [ ]:
# ev = "ENSG00000285043_ProteinCoding_1"
# rec = event_protein_map[ev]

In [ ]:
# # map interproscan results to their corresponding transcript (representing the splicing event of interest)
# analyses_to_exclude = ['NCBIFAM', 'SFLD']
# columns = ['protein_accession', 'sequence_length', 'analysis', 
#            'signature_description', 'start', 'stop', 'interpro_description']
# ipr = interproscan_results.loc[~interproscan_results['analysis'].isin(analyses_to_exclude), columns]

# ipr_grouped = {acc: grp for acc, grp in ipr.groupby('protein_accession')}

# event_interproscan_map = defaultdict(dict)

# rec_incl = rec['inclusion']
# incl_df  = ipr_grouped.get(rec_incl['transcript_id'])
# if incl_df is None:
#     continue
# overlap_df = incl_df[
#     (incl_df['start'] <= rec_incl['aa_end']) &
#     (incl_df['stop']  >= rec_incl['aa_start'])
# ]



# strand = rec['meta']['strand']

In [ ]:
# # junction siblings

# for sib in rec['exon_diff_junction_siblings']:
#     t = sib['transcript_id']
#     sib_df = ipr_grouped.get(t) 
#     if sib_df is None:
#         continue
#     sib_overlap = sib_df[
#         (sib_df['start'] <= sib['aa_end']) &
#         (sib_df['stop']  >= sib['aa_start'])
#     ]
#     break


In [ ]:
# # save AA position of protein domains in terms of genomic coordinates
# cds_obj = cds_by_transcript[t]
# cds_to_genome = rel_cds_to_genome(cds_obj, strand)
# genome_coords = []
# for idx, row in sib_overlap.iterrows():
#     aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
#     genome_coords.append(aa_to_genome)

### End debug

In [ ]:
len(interproscan_by_event)

12549

## Merge results with significant splicing info.

In [ ]:
with open('data/interproscan_by_event.pkl', "wb") as file:
    pickle.dump(interproscan_by_event, file)

In [ ]:
len(interproscan_by_event)

12549

In [ ]:
# merge cell type-specific splicing events with InterProScan results

interproscan_by_ct = dict() 
interproscan_by_ct_summary = dict()

uniprot_by_ct = dict()
uniprot_by_ct_summary = dict()

for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        ctype = file.split("_exons.csv")[0]
        print(ctype)
        
        signif_events_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        
        event_interproscan_dict = {
            ev: interproscan_by_event[ev] for ev in signif_events_df.index 
            if ev in interproscan_by_event
        }

        result = pd.concat(
            [df.assign(event_id=ev, bucket=bucket)
             for ev, buckets in event_interproscan_dict.items()
             for bucket, df in buckets.items()
             if bucket != 'meta' and df is not None],
            ignore_index=True
        )
        # move event and bucket to front
        cols = ['event_id', 'bucket'] + [c for c in result.columns if c not in ('event_id', 'bucket')]
        result = result[cols]
 
        # restrict to splicing events with InterProScan results
        df = result.merge(signif_events_df, left_on="event_id", right_index=True)
        interproscan_by_ct[ctype] = df
        
        # summarize interpro results per splicing event
        interproscan_by_ct_summary[ctype] = df.groupby(["event_id", "bucket"]).agg(
            r=('r', lambda x: ' | '.join(map(str, x.unique()))),
            is_specific=('is_specific', lambda x: ' | '.join(map(str, x.unique()))),
            Gene=('Gene', lambda x: ' | '.join(x.unique())),
            interproscan_id=('protein_accession', lambda x: ' | '.join(x.unique())),
            frame_preserving=('frame_preserving', lambda x: ' | '.join(map(str, x.unique()))),
            n_analyses=('analysis', lambda x: len(x.unique())),
            analyses=('analysis', lambda x: ' | '.join(x.unique())),
            signature_descriptions=('signature_description', lambda x: ' | '.join(str(v) for v in x.unique() if pd.notna(v))),
            interpro_descriptions=('interpro_description', lambda x: ' | '.join(x.unique()))
        ).reset_index()

Oligo
VLMC
Endo
Deep_layer_glutamatergic
Astro
OPC
Micro_PVM
All_Neuronal
All_GABAergic
Peri
CGE_Class
Upper_layer_glutamatergic


In [ ]:
with open("data/interproscan_by_ct.pkl", "wb") as file:
    pickle.dump(interproscan_by_ct, file)
    
with open("data/interproscan_by_ct_summary.pkl", "wb") as file:
    pickle.dump(interproscan_by_ct_summary, file)

In [ ]:
rec = interproscan_by_ct_summary['Deep_layer_glutamatergic']
rec[(rec['is_specific'] == "True") & (rec['bucket'] == "inclusion")].sort_values('n_analyses', ascending=False).head(10)

,event_id,bucket,r,is_specific,Gene,interproscan_id,frame_preserving,n_analyses,analyses,signature_descriptions,interpro_descriptions
533,ENSG00000056291_ProteinCoding_1,inclusion,0.4471791265064418,True,NPFFR2,ENST00000308744,False,12,CATH-Gene3D | CATH-FunFam | CDD | PIRSR | Pfam...,Rhodopsin 7-helix transmembrane proteins | neu...,"- | G protein-coupled receptor, rhodopsin-like..."
5152,ENSG00000146904_ProteinCoding_1,inclusion,0.3414524560565585,True,EPHA1,ENST00000275815,True,12,CATH-Gene3D | CATH-FunFam | CDD | PIRSR | Pfam...,"Transcription Factor, Ets-1 | Phosphorylase Ki...",Sterile alpha motif/pointed domain superfamily...
199,ENSG00000010704_ProteinCoding_7,inclusion,-0.1190834969825156,True,HFE,ENST00000353147,True,11,CATH-Gene3D | CATH-FunFam | Pfam | Phobius | S...,Immunoglobulins | Major histocompatibility com...,Immunoglobulin-like fold | - | Immunoglobulin ...
4616,ENSG00000139880_ProteinCoding_1,inclusion,-0.4879409185382066,True,CDH24,ENST00000397359,True,11,CATH-Gene3D | CATH-FunFam | CDD | Pfam | Phobi...,Cadherins | Cadherin 24 | Protocadherin beta 4...,- | Cadherin-like | Cadherin-like superfamily ...
275,ENSG00000017260_ProteinCoding_4,inclusion,0.2051146889237357,True,ATP2C1,ENST00000510168,True,10,CATH-Gene3D | CATH-FunFam | Pfam | Phobius | S...,"Calcium-transporting ATPase, transmembrane dom...","- | P-type ATPase, A domain | Cation-transport..."
6688,ENSG00000167618_ProteinCoding_2,inclusion,0.3080313269016527,True,LAIR2,ENST00000301202,True,10,CATH-Gene3D | CATH-FunFam | MobiDB-lite | Pfam...,Immunoglobulins | Leukocyte immunoglobulin-lik...,Immunoglobulin-like fold | - | Immunoglobulin ...
7231,ENSG00000176884_ProteinCoding_1,inclusion,0.3713913965532838,True,GRIN1,ENST00000371560,True,10,CATH-Gene3D | CATH-FunFam | Phobius | CDD | PI...,"- | glutamate receptor ionotropic, NMDA 1 isof...","- | Glutamate [NMDA] receptor subunit 1-like, ..."
7265,ENSG00000177508_ProteinCoding_1,inclusion,-0.1988403026764951,True,IRX3,ENST00000329734,False,10,CATH-Gene3D | CATH-FunFam | COILS | CDD | Mobi...,Homeodomain-like | Iroquois-class homeobox pro...,- | Homeodomain | KN homeodomain | Iroquois-cl...
4027,ENSG00000134207_ProteinCoding_1,inclusion,-0.1585543203672459,True,SYT6,ENST00000610222,True,9,CATH-Gene3D | CATH-FunFam | CDD | PIRSR | Pfam...,C2 domain | Synaptotagmin 6 | C2 domain second...,C2 domain superfamily | - | C2 domain
5132,ENSG00000146122_ProteinCoding_1,inclusion,0.2245671444135169,True,DAAM2,ENST00000633794,True,8,CATH-FunFam | COILS | MobiDB-lite | Pfam | SMA...,Dishevelled associated activator of morphogene...,"- | Formin, FH2 domain | Formin, FH2 domain su..."
